## plan
use SAGE modelling method (t-SNE into DBSCAN)

### import testing

In [ ]:
%pip install cudf-cu13==26.2.* cuml-cu13==26.2.* 

In [ ]:
%pip list -v

In [ ]:
!df /apps/jasmin/jaspy/miniforge_envs/jaspy3.12/mf3-25.3.0-3/envs/jaspy3.12-mf3-25.3.0-3-v20250704/ -h

In [ ]:
!df /home  -h

In [ ]:
%pip install ipympl

In [ ]:
# %load_ext cuml.accel

from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.cluster import DBSCAN
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr

In [ ]:
## CONSTANTS

data_path = './data/ACOD_CTD_v1.0.nc'

### data cleaning (testing)

In [ ]:
def null_frac(da):

    return da.isnull().sum().data / da.size

In [ ]:
full_data = xr.open_dataset('./data/ACOD_CTD_v1.0.nc')

In [ ]:
top_layer = full_data.sel(PRESSURE=slice(0,50))

print(null_frac(full_data.Temperature))
print(null_frac(top_layer.Temperature))

In [ ]:
top_temp = top_layer.Temperature
itp_temp = top_layer.Temperature.interpolate_na(dim='PRESSURE')
null_frac(itp_temp)

In [ ]:
remasked = top_temp.where(~top_temp.sel(PRESSURE=50).isnull())
itp_remasked = remasked.interpolate_na(dim='PRESSURE')

null_frac(itp_remasked)

In [ ]:
depths = list(range(10, 1000, 10))

null_fracs = []
for i in depths:
    top_layer = full_data.Temperature.sel(PRESSURE=slice(0,i))
    null_fracs.append(null_frac(top_layer))

plt.plot(depths, null_fracs)

In [ ]:
depths = list(range(10, 500, 10))

null_fracs = []
itp_nf = []
for i in depths:
    top_layer = full_data.Temperature.sel(PRESSURE=slice(0,i))
    itp_tl = top_layer.interpolate_na(dim='PRESSURE')
    null_fracs.append(null_frac(top_layer))
    itp_nf.append(null_frac(itp_tl))

import matplotlib.pyplot as plt

plt.plot(depths, null_fracs, label='original')
plt.plot(depths, itp_nf, label='interpolated')

plt.legend()

In [ ]:
depths = list(range(10, 80))

null_fracs = []
itp_nf = []
for i in depths:
    top_layer = full_data.Temperature.sel(PRESSURE=slice(0,i))
    itp_tl = top_layer.interpolate_na(dim='PRESSURE')
    null_fracs.append(null_frac(top_layer))
    itp_nf.append(null_frac(itp_tl))

import matplotlib.pyplot as plt

plt.plot(depths, null_fracs, label='original')
plt.plot(depths, itp_nf, label='interpolated')

plt.legend()

### data cleaning (proper)

In [ ]:
data = xr.open_dataset(data_path)
upper_data = data.sel(PRESSURE=slice(0, 50))

tot_profiles = len(upper_data.PROFILE)

data

In [ ]:
temp_array = upper_data.interpolate_na(dim='PRESSURE').Temperature.data
sal_array = upper_data.interpolate_na(dim='PRESSURE').Salinity_PSS_with_QC_applied

'''for i in range(tot_profiles):
    print(np.sum(np.isnan(temp_array[:, i])))'''

temp_nan_mask = ~np.sum(np.isnan(temp_array),axis=0).astype(bool)
sal_nan_mask = ~np.sum(np.isnan(sal_array),axis=0).astype(bool)

nan_mask = temp_nan_mask & sal_nan_mask

# originally 29717 values; ended up with 23604
clean_temperature = temp_array[:,nan_mask].T
clean_salinity = sal_array[:,nan_mask].T

masked_data = data.sel(PROFILE=nan_mask)

clean_temperature.shape
## TODO: repeat with salinity and intersect masks to get final cleaned dataset

### normalising data

In [ ]:
## should be in the shape (n_samples, n_variables) where variables here are pressure levels
num_samples = clean_temperature.shape[0]

temp_mean = clean_temperature.mean(axis=0)
ptwise_temp_mean = np.tile(temp_mean, (num_samples, 1))

temp_std = clean_temperature.std(axis=0)
ptwise_temp_std = np.tile(temp_std, (num_samples, 1))

normalised_temp = (clean_temperature - ptwise_temp_mean) / ptwise_temp_std

## TODO: repeat for salinity (lowkey just whack salinity on the end and rename variables is fine)

## the model

## dimensionality reduction (PCA)

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=4)
pca.fit(normalised_temp)

plt.plot(range(1, pca.n_components_+1), pca.explained_variance_ratio_, marker='o')
plt.grid(True)

In [ ]:
reduced_temp = pca.fit_transform(normalised_temp)

reduced_temp.shape

## the models

### GMM

In [ ]:
from sklearn.mixture import GaussianMixture

GMM = GaussianMixture(n_components=5)

clusters = GMM.fit_predict(normalised_temp)

## Plots

In [ ]:
import matplotlib as mpl

def dec_to_hex(n):
    '''
    for use in rgba_to_hex

    scales numbers from 0 to 1 -> hex from 1 -> 225
    '''

    if n == 1:
        return 'FF'
    
    decimal_value = round(n * 256)
    hex_val = f'{decimal_value:02x}'

    return hex_val

def rgba_to_hex(arr):
    '''
    assuming input is in form from colormap
    (r, g, b, a)
    '''

    rgb_values = arr[:3]

    return '#' + ''.join([dec_to_hex(num) for num in rgb_values])

cmap = mpl.colormaps['Set1']
_cid = np.linspace(0, 1, len(set(clusters)))

colors = [
    rgba_to_hex(cmap(i)) for i in _cid
]

pt_clrs = list(map(lambda i: colors[i], clusters))

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature

lats = masked_data.LATITUDE
lons = masked_data.LONGITUDE

fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180)})

ax.add_feature(cfeature.COASTLINE)
ax.add_feature(cfeature.BORDERS)
ax.add_feature(cfeature.LAND)

ax.scatter(lons+180, lats, s=1, alpha=0.4, c=pt_clrs)

In [ ]:
set(clusters)

In [ ]:
## plot masked data



import cartopy.crs as ccrs
import cartopy.feature as cfeature

full_lats = masked_data.LATITUDE
full_lons = masked_data.LONGITUDE



fig, axs = plt.subplots(5, 1, figsize=(15, 20), subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180)})
axs = axs.flatten()

for cluster_id in range(5):
    ax = axs[cluster_id]
    cluster_mask = clusters == cluster_id
    
    cluster_lats = full_lats[cluster_mask]
    cluster_lons = full_lons[cluster_mask]
    
    ax.add_feature(cfeature.COASTLINE)
    ax.add_feature(cfeature.BORDERS)
    ax.add_feature(cfeature.LAND)
    
    ax.scatter(lons+180, lats, s=1, alpha=0.4, c='Black')
    ax.scatter(cluster_lons+180, cluster_lats, s=1, alpha=0.7, c='Red')

    ax.set_title(f'Cluster {cluster_id}')

# old crummy code

### code (v1)

In [ ]:
n_test_samples = 5000

t_SNE_embedding = TSNE(n_components=3)

test_temp = clean_temperature[:,:n_test_samples]
test_sal = clean_salinity[:,:n_test_samples]

test_data = np.concatenate([test_temp, test_sal], axis=0)

In [ ]:
temp_embedded = t_SNE_embedding.fit_transform(test_data.T)

In [ ]:
fig, ax = plt.subplots(subplot_kw={'projection': '3d'})

x, y, z = temp_embedded.T
ax.scatter(x, y, z)

In [ ]:
cluster_DBSCAN = DBSCAN(eps=0.5, min_samples=5)
clustering = cluster_DBSCAN.fit(temp_embedded)

In [ ]:
clusters = clustering.labels_
set(clusters)

### plotting

In [ ]:
import matplotlib as mpl

def dec_to_hex(n):
    '''
    for use in rgba_to_hex

    scales numbers from 0 to 1 -> hex from 1 -> 225
    '''

    decimal_value = round(n * 256)

    return f'{decimal_value:02x}'

def rgba_to_hex(arr):
    '''
    assuming input is in form from colormap
    (r, g, b, a)
    '''

    rgb_values = arr[:3]

    return '#' + ''.join([dec_to_hex(num) for num in rgb_values])

cmap = mpl.colormaps['viridis']
_cid = np.linspace(0, 1, len(set(clusters)))

colors = [
    rgba_to_hex(cmap(i)) for i in _cid
]

In [ ]:
pt_clrs = list(map(lambda i: colors[i], clustering.labels_))
lons = masked_data.LONGITUDE[:n_test_samples]
lats = masked_data.LATITUDE[:n_test_samples]

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature

fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180)})

ax.add_feature(cfeature.COASTLINE)
ax.add_feature(cfeature.BORDERS)
ax.add_feature(cfeature.LAND)

ax.scatter(lons+180, lats, c=pt_clrs, s=1)

In [ ]:
cluster_mask = clusters == -1

cluster_data = masked_data.isel(PROFILE=slice(0, n_test_samples), PRESSURE=slice(0, 51)).sel(PROFILE=cluster_mask)

fig, ax = plt.subplots()

cluster_temp = cluster_data.Temperature
cluster_sal = cluster_data.Salinity_PSS_with_QC_applied
depth_vals = cluster_data.PRESSURE

for i in range(len(cluster_temp.PROFILE)):

    profile = cluster_temp.isel(PROFILE=i)

    ax.plot(profile, -depth_vals, alpha=0.01, color='blue')

ax.set_ylim(-50, 0)

### the GMM bit

In [ ]:
pca_decompose = PCA(n_components = 3)

test_embedded = pca_decompose.fit_transform(test_data.T)

fig, ax = plt.subplots(subplot_kw={'projection': '3d'})

x, y, z = test_embedded.T
ax.scatter(x, y, z)

In [ ]:
GMM = GaussianMixture(n_components=3)
transformed = GMM.fit_predict(test_embedded)

import cartopy.crs as ccrs
import cartopy.feature as cfeature

fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180)})

ax.add_feature(cfeature.COASTLINE)
ax.add_feature(cfeature.BORDERS)
ax.add_feature(cfeature.LAND)

import matplotlib as mpl

cmap = mpl.colormaps['viridis']
_cid = np.linspace(0, 1, len(set(transformed)))

colors = [
    rgba_to_hex(cmap(i)) for i in _cid
]

pt_clrs = list(map(lambda i: colors[i], transformed))
lons = masked_data.LONGITUDE[:n_test_samples]
lats = masked_data.LATITUDE[:n_test_samples]

ax.scatter(lons+180, lats, c=pt_clrs, s=1)

In [ ]:
set(transformed)

In [ ]:
cluster_mask = transformed == 2

cluster_data = masked_data.isel(PROFILE=slice(0, n_test_samples), PRESSURE=slice(0, 51)).sel(PROFILE=cluster_mask)

fig, ax = plt.subplots()

cluster_temp = cluster_data.Temperature
depth_vals = cluster_data.PRESSURE

for i in range(len(cluster_temp.PROFILE)):

    profile = cluster_temp.isel(PROFILE=i)

    ax.plot(profile, -depth_vals, alpha=0.05, color='blue')

ax.set_ylim(-50, 0)

In [ ]:
set(transformed)